In [1]:

# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import requests
import re
from bs4 import BeautifulSoup


import datetime

import pandas as pd

from pandas import ExcelWriter

from time import sleep



import os




In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'BD IDRABD' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.4")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


Running BD IDRABD Web Scraping Tool v.1.4


In [3]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


def extract_address_until_empty(address_list):
    cleaned = []
    for item in address_list:
        if item.strip() == '':
            break  # Stop at the first empty or whitespace-only element
        cleaned.append(item.strip())
    return cleaned


def extract_city_zip(address):
    # Match city names followed by a dash or space and 4–5 digit ZIP code
    # Ensure city name is not preceded by a single letter or word
    match = re.search(r'(?:^|,\s*)([A-Za-z\s]+?)[\s\-]+(\d{4,5})\b', address)
    if match:
        city = match.group(1).strip()
        zip_code = match.group(2)
        return city, zip_code
    return '', ''



In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={	

#'BD IDRABD 1': 'https://idra.org.bd/site/page/ab3f33d4-fd9d-478b-ad0c-abc27ad93cfe/-', 

'BD IDRABD 1': 'https://idra.org.bd/pages/static-pages/6922e04a933eb65569e265aa',

'BD IDRABD 2': 'https://idra.org.bd/site/page/450e34d8-5aff-4e7a-a956-67bb50dc2c9c/-', 

'BD IDRABD 3': 'https://idra.org.bd/site/page/6dba4800-86e2-4304-9271-2c2f16b5e9e1/-', 

'BD IDRABD 4': 'https://idra.org.bd/site/page/c25211d9-9cbc-4dd6-bdc9-50a5dec82793/-', 

 'BD IDRABD 5': 'https://idra.org.bd/site/page/37e6fd51-f4fb-4d57-aa6d-32a13e7b701e/-', 



}

Typology={
'BD IDRABD 1':	'List of Life Insurers',
'BD IDRABD 2':	'List of Non-Life Insurers',
'BD IDRABD 3':  'List of Life Bancassurance Institutions',
'BD IDRABD 4':	'List of Non-Life Bancassurance Institutions',
'BD IDRABD 5':	'List of Authorized Insurtech Institutions',

}

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

	  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}



processdate = now.strftime('%Y-%m-%d')



# %%


In [5]:


payload = {
    'lang': 'en',
}
headers = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36 Edg/141.0.0.0'
}

# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):    

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")

    response = requests.post(regdict[reg], data=payload, verify=False, headers=headers)

    try:
        json_data = response.json()
        print("Response is JSON:")
        print(json_data)
    except ValueError:
        print("Response is not in JSON format.")
        soup = response.text




    # Assuming `response.text` contains the HTML content
    soup = BeautifulSoup(response.text, 'html.parser')


    for br in soup.find_all('br'):
        br.replace_with('\n')


    # Now you can search for tables
    tables = soup.find_all('table')
    table = tables [0]

    tbody = table.find('tbody')


    if reg == 'BD IDRABD 1' or reg == 'BD IDRABD 2':

        if reg == 'BD IDRABD 1':
            tds = tbody.find_all('td', style=re.compile(r'width:\s*180px'))
        elif reg =='BD IDRABD 2':
            tds = tbody.find_all('td', style=re.compile(r'width:\s*246px'))

        for i in range(1,len(tds)):

            #print('Name : ',tds[i].find('p').text.replace('\n',''))
            # print('Adress: ',tds[i].find_all('p')[1].text.replace('\n',''))
            #print('Website:', tds[i].find_all('p')[-1].text.replace('\n',''))
            total_info = tds[i].get_text(separator='\n').strip().strip()
            total_info = re.sub(r'[\xa0\r\t]', ' ', total_info)
            #print(i,total_info.split('\n'))
            
            name = total_info.split('\n')[0]
            #print('Name: ', name)
            sqldict['Name'].append(name)
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegulationType'].append('Regulated')
            
            
            rest = ' '.join(total_info.split('\n')[1:]).strip()
            emails = re.findall(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', rest)
            email = emails[0] if emails and emails[0] != '' else ''
            #print('Email:', emails[0] if emails and emails[0] != '' else '')
            sqldict['Email'].append(email)


            phones = re.findall(r'\b(?:\+?88)?0\d{9,10}\b|\b\d{2,}-?\d{6,}\b', rest)
            phone = phones[0] if phones and phones[0] != '' else ''
            #print('Phone: ',phones[0] if phones and phones[0]!='' else '')
            sqldict['Phone'].append(phone)

            # Extract websites
            websites = re.findall(r'\b(?:https?://)?(?:www\.)?[\w\-]+\.\w{2,}(?:\.\w{2,})?\b', rest)
            # print('Website:', websites[0] if websites else '')
                
            cleaned_rest = rest
            for email in emails:
                cleaned_rest = cleaned_rest.replace(email, '')
            for phone in phones:
                cleaned_rest = cleaned_rest.replace(phone, '')
            for website in websites:
                cleaned_rest = cleaned_rest.replace(website, '')

            address_temp = extract_address_until_empty(cleaned_rest.split('  '))
            if any(x in address_temp[-1] for x in ['@', ':', 'www']):
                address_temp = address_temp[:-1]  # You probably meant to slice or modify here
            else:
                pass

            
            address_temp = ' '.join(address_temp)
            address_temp = address_temp if len(address_temp)>10  else ''
            # print('Address: ', address_temp)
            sqldict['Address_1'].append(address_temp)

            city, zip_code = extract_city_zip(address_temp)
            # print(city,zip_code)
            # print('')
            sqldict['City'].append(city)
            sqldict['Zip'].append(zip_code)
            sqldict['ListProcessDate'].append(processdate)
            sqldict = bourange_same_length_array(sqldict)

    elif reg == 'BD IDRABD 3':
        #print(tbody)
        
            trs = tbody.find_all('tr')
            for index,tr in enumerate(trs):
                if index ==0:
                    continue
                tds = tr.find_all('td')

                name = tds[1].text.strip()
                license_num = tds[3].text.replace('\n',' ').strip()
                issu_date = tds[4].text.strip()
                phone = tds[5].text.strip()

                sqldict['Name'].append(name)
                sqldict['InternalID_1'].append(license_num)
                sqldict['InternalID_1_type'].append('License Number')
                sqldict['RegulationDate'].append(issu_date.replace('\n',' ').strip())
                sqldict['Phone'].append(phone)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)
    elif reg == 'BD IDRABD 4':
        trs = tbody.find_all('tr')
        for index,tr in enumerate(trs):
            if index ==0:
                continue
            tds = tr.find_all('td')
            name = tr.find('td', style=re.compile(r'width:\s*145px'))
            license_num = tr.find('td', style=re.compile(r'width:\s*78px'))
            issu_date_ = tr.find('td', style=re.compile(r'width:\s*102px'))
            hotline = tr.find('td', style=re.compile(r'width:\s*97px'))
            try:
                name = name.text.strip()
                license = license_num.text.strip()
                issu_date = issu_date_.text.strip()
                phone = hotline.text.strip()
                print(license)
                sqldict['Name'].append(name)
                sqldict['InternalID_1'].append(license)
                sqldict['InternalID_1_type'].append('License Number')
                sqldict['RegulationDate'].append(issu_date)
                sqldict['Phone'].append(phone)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)

            except:
                pass

    elif reg == 'BD IDRABD 5':
            trs = tbody.find_all('tr')
            for index,tr in enumerate(trs):
                if index ==0:
                    continue
                tds = tr.find_all('td')

                name = tds[1].text.strip()
                address =  tds[2].text.strip()
                phone = tds[4].text.strip()
                email_ = tds[5].text.strip()

                sqldict['Name'].append(name)
                sqldict['Phone'].append(phone)
                sqldict['Email'].append(email_)
                sqldict['Address_1'].append(address)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)



[INFO] : Working 1/5 _(BD IDRABD 1)_ 


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'idra.org.bd'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Response is not in JSON format.


IndexError: list index out of range

In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)




C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_31232\3985938601.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
df.to_csv('total_version_4.csv')